# 10年定着予測 - 手動CatBoostアブレーション & AutoGluon HPO拡張

**目的**: `13_autogluon_prototype.ipynb`の結果（CatBoost単体 Public 0.574165、全提出中最良）を受けて、
`submit_result_report.md`セクション11で提案した次のアクションのうち、以下3点を検証する。

## 検証する3つの疑問

1. **なぜAutoGluonの単体CatBoostが12_の手動CatBoost（OOF 0.5868）を大きく上回ったのか**
   → 13_と全く同じ特徴量・全く同じ検証split（時系列ホールドアウト）の上で、
   自前のCatBoost実装（Optunaでチューニング）を学習し、AutoGluonのCatBoost（検証Log Loss 0.567941）と比較する（実験A）。
   もし自前実装が同水準に到達すれば「検証方式と特徴量の単純化」が主因、届かなければ
   AutoGluon固有のゼロショット既定パラメータや内部前処理が効いていることになる。

2. **`time_limit`を伸ばす、またはHPOを追加すればさらに改善するか**
   → 内部バギング（`num_bag_folds`）は引き続き無効化したまま（過学習リスクの再燃を避けるため）、
   CatBoostに複数のハイパーパラメータ候補を明示的に与えつつ`time_limit`を伸ばして再学習する（実験B）。

3. **単一時系列ホールドアウトがTimeSeriesSplit OOFよりPublicとの整合性が高いという観察は再現するか**
   → 実験Aの自前CatBoostも13_と同じ単一ホールドアウトで検証するため、この検証方式が
   AutoGluon以外のモデルでも同様に機能するかの追加データ点になる。

## 前回からの運用改善（次のアクション4への対応）

13_では「単体最良モデル」の提出ファイルを後から追加しようとして、Colab/Drive間の同期問題で
一度失敗した。今回は**最初からアンサンブル(Weighted Ensemble)と単体最良モデルの両方を必ず保存する**
構成にしておく。

## 実行環境
Google Colab（GPU: T4、ハイメモリ）を想定。特徴量エンジニアリングは`13_`と完全に同一のものを流用する
（比較の公平性を保つため、あえて変更しない）。

In [40]:
# AutoGluon (Tabular, NN系モデルを含むフル版) のインストール
!pip install -q autogluon.tabular[all]

In [41]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Aug  8 16:07:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             29W /   70W |     295MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import datetime
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from autogluon.tabular import TabularPredictor

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [44]:
SCRIPT_NAME = "14_catboost_ablation_autogluon_hpo_bk"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_MANUAL_CAT_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_manual_catboost_holdout.csv"
SUBMISSION_AG_ENSEMBLE_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_autogluon_ensemble.csv"
SUBMISSION_AG_SINGLE_BEST_PATH_TEMPLATE = str(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_autogluon_single_best_{{model}}.csv")

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
AUTOGLUON_MODEL_DIR = SAVED_MODELS_DIR / "autogluon_hpo"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"AutoGluon Model Directory: {AUTOGLUON_MODEL_DIR}")

[2026-08-08 16:07:42] [INFO] === [14_catboost_ablation_autogluon_hpo_bk] 実験開始 ===


INFO:14_catboost_ablation_autogluon_hpo_bk:=== [14_catboost_ablation_autogluon_hpo_bk] 実験開始 ===


[2026-08-08 16:07:42] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


INFO:14_catboost_ablation_autogluon_hpo_bk:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


[2026-08-08 16:07:42] [INFO] AutoGluon Model Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/14_catboost_ablation_autogluon_hpo_bk/autogluon_hpo


INFO:14_catboost_ablation_autogluon_hpo_bk:AutoGluon Model Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/14_catboost_ablation_autogluon_hpo_bk/autogluon_hpo


In [45]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-08 16:07:43] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:14_catboost_ablation_autogluon_hpo_bk:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-08 16:07:43] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:14_catboost_ablation_autogluon_hpo_bk:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-08 16:07:43] [INFO] 定着率: 0.5647


INFO:14_catboost_ablation_autogluon_hpo_bk:定着率: 0.5647


[2026-08-08 16:07:43] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:14_catboost_ablation_autogluon_hpo_bk:Train IDs: 2761, Test IDs: 2502


## 1. 特徴量エンジニアリング（13_と完全に同一）

実験A・実験Bともに「13_と同じ特徴量」で比較するため、13_の特徴量生成コードをそのまま流用する
（LabelEncoding・多項式・ビン化・複合カテゴリ交互作用は使わない）。

In [46]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_/13_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次集約・カテゴリ変化・欠損値・ドメイン特徴量関数定義完了")

✅ 月次集約・カテゴリ変化・欠損値・ドメイン特徴量関数定義完了


In [47]:
def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_department_target_encoding(train_persona, test_persona, y_train, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding（12_/13_と同一ロジック）"""
    col = "初期部署ID"
    global_mean = y_train.mean()

    train_te = np.zeros(len(train_persona))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    dept_train = train_persona[col].values
    y_arr = y_train.values

    for tr_idx, val_idx in kf.split(train_persona):
        df_tr = pd.DataFrame({col: dept_train[tr_idx], "y": y_arr[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        train_te[val_idx] = pd.Series(dept_train[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_train, "y": y_arr})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        "社員ID": train_persona["社員ID"].values,
        "dept_target_enc": train_te,
        "dept_size": train_persona[col].map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        "社員ID": test_persona["社員ID"].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_/13_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_/13_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ 高度統計・クラスター・部署Target Encoding・EDA駆動・上司チーム規模の関数定義完了")

✅ 高度統計・クラスター・部署Target Encoding・EDA駆動・上司チーム規模の関数定義完了


In [48]:
logger.info("-" * 60)
logger.info("特徴量生成開始")
logger.info("-" * 60)

logger.info("月次集約特徴量を生成中...")
train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

logger.info("月次カテゴリ変化特徴量を生成中...")
train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

logger.info("欠損値特徴量を生成中...")
train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

logger.info("ドメイン知識特徴量を生成中...")
train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

logger.info("高度な統計特徴量を生成中...")
train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

logger.info("クラスター特徴量を生成中...")
train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

logger.info("部署ID Target Encodingを生成中...")
train_dept_te, test_dept_te = create_department_target_encoding(
    train_persona, test_persona, y_train, seed=SEED, n_splits=5, smoothing=10
)

logger.info("EDA駆動特徴量（欠勤パターン・評価タイミング・ボラティリティ・比率）を生成中...")
train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

logger.info("上司チーム規模特徴量を生成中...")
train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("特徴量生成完了")

[2026-08-08 16:07:43] [INFO] ------------------------------------------------------------


INFO:14_catboost_ablation_autogluon_hpo_bk:------------------------------------------------------------


[2026-08-08 16:07:43] [INFO] 特徴量生成開始


INFO:14_catboost_ablation_autogluon_hpo_bk:特徴量生成開始


[2026-08-08 16:07:43] [INFO] ------------------------------------------------------------


...

INFO:14_catboost_ablation_autogluon_hpo_bk:EDA駆動特徴量（欠勤パターン・評価タイミング・ボラティリティ・比率）を生成中...


[2026-08-08 16:14:16] [INFO] 上司チーム規模特徴量を生成中...


INFO:14_catboost_ablation_autogluon_hpo_bk:上司チーム規模特徴量を生成中...


[2026-08-08 16:14:16] [INFO] 特徴量生成完了


INFO:14_catboost_ablation_autogluon_hpo_bk:特徴量生成完了


In [49]:
logger.info("テキスト特徴量を生成中...")
text_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
train_persona["text_total_chars"] = train_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
for col in text_cols:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)

logger.info("時間的特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

logger.info("交互作用特徴量を生成中...")
train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("特徴量処理完了")

[2026-08-08 16:14:16] [INFO] テキスト特徴量を生成中...


INFO:14_catboost_ablation_autogluon_hpo_bk:テキスト特徴量を生成中...


[2026-08-08 16:14:16] [INFO] 時間的特徴量を生成中...


INFO:14_catboost_ablation_autogluon_hpo_bk:時間的特徴量を生成中...


[2026-08-08 16:14:16] [INFO] 交互作用特徴量を生成中...


INFO:14_catboost_ablation_autogluon_hpo_bk:交互作用特徴量を生成中...


[2026-08-08 16:14:16] [INFO] 特徴量処理完了


INFO:14_catboost_ablation_autogluon_hpo_bk:特徴量処理完了


In [50]:
logger.info("-" * 60)
logger.info("特徴量の統合")
logger.info("-" * 60)

train_persona_features = train_persona.drop(columns=[TARGET_COL])

train_features = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
train_features = train_features.merge(train_cat_change, on=ID_COL, how="left")
train_features = train_features.merge(train_missing, on=ID_COL, how="left")
train_features = train_features.merge(train_domain, on=ID_COL, how="left")
train_features = train_features.merge(train_advanced_stats, on=ID_COL, how="left")
train_features = train_features.merge(train_cluster, on=ID_COL, how="left")
train_features = train_features.merge(train_dept_te, on=ID_COL, how="left")
train_features = train_features.merge(train_eda_feats, on=ID_COL, how="left")
train_features = train_features.merge(train_mgr, on=ID_COL, how="left")

test_features = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
test_features = test_features.merge(test_cat_change, on=ID_COL, how="left")
test_features = test_features.merge(test_missing, on=ID_COL, how="left")
test_features = test_features.merge(test_domain, on=ID_COL, how="left")
test_features = test_features.merge(test_advanced_stats, on=ID_COL, how="left")
test_features = test_features.merge(test_cluster, on=ID_COL, how="left")
test_features = test_features.merge(test_dept_te, on=ID_COL, how="left")
test_features = test_features.merge(test_eda_feats, on=ID_COL, how="left")
test_features = test_features.merge(test_mgr, on=ID_COL, how="left")

logger.info(f"Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-08 16:14:16] [INFO] ------------------------------------------------------------


INFO:14_catboost_ablation_autogluon_hpo_bk:------------------------------------------------------------


[2026-08-08 16:14:16] [INFO] 特徴量の統合


INFO:14_catboost_ablation_autogluon_hpo_bk:特徴量の統合


[2026-08-08 16:14:16] [INFO] ------------------------------------------------------------


INFO:14_catboost_ablation_autogluon_hpo_bk:------------------------------------------------------------


[2026-08-08 16:14:16] [INFO] Train: (2761, 315), Test: (2502, 315)


INFO:14_catboost_ablation_autogluon_hpo_bk:Train: (2761, 315), Test: (2502, 315)


In [51]:
logger.info("職種別の乖離特徴量を生成中...")
job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
job_means = {m: train_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
category_means_train = {m: train_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}

for df in [train_features, test_features]:
    for m in job_dev_metrics:
        job_mean_series = df["初期職種"].map(job_means[m])
        df[f"{m}_job_deviation"] = df[m] - job_mean_series
    df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
    df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)

logger.info("給与の相対化特徴量を生成中...")
grade_salary_mean = train_features.groupby("初期等級")["初任給_円"].mean().to_dict()
category_salary_mean = train_features.groupby("入社区分")["初任給_円"].mean().to_dict()
grade_monthly_salary_mean = train_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

for df in [train_features, test_features]:
    df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
    df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
    df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

logger.info(f"派生特徴量生成後 Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-08 16:14:17] [INFO] 職種別の乖離特徴量を生成中...


INFO:14_catboost_ablation_autogluon_hpo_bk:職種別の乖離特徴量を生成中...


[2026-08-08 16:14:17] [INFO] 給与の相対化特徴量を生成中...


INFO:14_catboost_ablation_autogluon_hpo_bk:給与の相対化特徴量を生成中...


[2026-08-08 16:14:17] [INFO] 派生特徴量生成後 Train: (2761, 323), Test: (2502, 323)


INFO:14_catboost_ablation_autogluon_hpo_bk:派生特徴量生成後 Train: (2761, 323), Test: (2502, 323)


In [52]:
# 不要な列の削除（13_と同一）
drop_cols = [
    "入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
    "初期部署ID", "初期等級", "最終学歴", "前職職種",
]
drop_cols_exist = [col for col in drop_cols if col in train_features.columns]
train_features = train_features.drop(columns=drop_cols_exist)
test_features = test_features.drop(columns=drop_cols_exist)

train_features = train_features.set_index(ID_COL)
test_features = test_features.set_index(ID_COL)

logger.info(f"最終特徴量数: {train_features.shape[1]} (Train: {train_features.shape}, Test: {test_features.shape})")

[2026-08-08 16:14:17] [INFO] 最終特徴量数: 315 (Train: (2761, 315), Test: (2502, 315))


INFO:14_catboost_ablation_autogluon_hpo_bk:最終特徴量数: 315 (Train: (2761, 315), Test: (2502, 315))


## 2. 時系列ホールドアウトの作成（13_と完全に同一split）

実験A・実験Bとも同じ`ag_train_data`/`ag_tuning_data`を使うことで、公平な比較にする。

In [53]:
target_series = train_persona.set_index(ID_COL)[TARGET_COL]

train_features_sorted = train_features.sort_values("入社日")
y_sorted = target_series.loc[train_features_sorted.index]

split_point = int(len(train_features_sorted) * 0.8)
ag_train_data = train_features_sorted.iloc[:split_point].copy()
ag_tuning_data = train_features_sorted.iloc[split_point:].copy()
ag_train_data[TARGET_COL] = y_sorted.iloc[:split_point].values
ag_tuning_data[TARGET_COL] = y_sorted.iloc[split_point:].values

logger.info(f"train_data: {ag_train_data.shape} (入社日 {ag_train_data['入社日'].min().date()} 〜 {ag_train_data['入社日'].max().date()})")
logger.info(f"tuning_data: {ag_tuning_data.shape} (入社日 {ag_tuning_data['入社日'].min().date()} 〜 {ag_tuning_data['入社日'].max().date()})")
logger.info(f"train定着率: {ag_train_data[TARGET_COL].mean():.4f}, tuning定着率: {ag_tuning_data[TARGET_COL].mean():.4f}")

[2026-08-08 16:14:17] [INFO] train_data: (2208, 316) (入社日 2011-04-01 〜 2013-04-01)


INFO:14_catboost_ablation_autogluon_hpo_bk:train_data: (2208, 316) (入社日 2011-04-01 〜 2013-04-01)


[2026-08-08 16:14:17] [INFO] tuning_data: (553, 316) (入社日 2013-04-01 〜 2014-03-01)


INFO:14_catboost_ablation_autogluon_hpo_bk:tuning_data: (553, 316) (入社日 2013-04-01 〜 2014-03-01)


[2026-08-08 16:14:17] [INFO] train定着率: 0.5616, tuning定着率: 0.5769


INFO:14_catboost_ablation_autogluon_hpo_bk:train定着率: 0.5616, tuning定着率: 0.5769


## 3. 実験A: 手動CatBoost（Optuna）を13_と全く同じ特徴量・検証splitで学習

**疑問1への回答**: AutoGluonの`common/catboost/cat_model_v2.py`は`cv_strategy`が`'stratified'`か`'timeseries'`
（TimeSeriesSplitの多fold）のみに対応しており、13_のような「単一の固定ホールドアウト」には対応していない。
そのため、この実験だけは`common/`のラッパーを使わず、CatBoostを直接呼び出してOptunaでチューニングする
（`ag_train_data`で学習・`ag_tuning_data`で検証、という13_と完全に同じ条件）。

**注意（比較の非対称性）**: AutoGluonは生の`入社日`（datetime型）も特徴量として使い自動で年月日等に分解するが、
CatBoostのPythonライブラリはdatetime型を直接扱えないため、本実験では`入社日`列を除外する
（代わりに`入社年`/`入社月`/`入社四半期`という分解済みの特徴量は含まれている）。この差は結果解釈の際に考慮する。

In [54]:
# 特徴量列の準備（入社日はCatBoostが直接扱えないため除外。入社年/月/四半期で代替）
feature_cols = [c for c in train_features.columns if c != "入社日"]
cat_cols = [c for c in feature_cols if train_features[c].dtype == "object"]
logger.info(f"実験A 特徴量数: {len(feature_cols)}, カテゴリ列: {cat_cols}")

X_tr = ag_train_data[feature_cols].fillna(-999)
y_tr = ag_train_data[TARGET_COL]
X_va = ag_tuning_data[feature_cols].fillna(-999)
y_va = ag_tuning_data[TARGET_COL]

def objective_manual_cat(trial):
    params = {
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "iterations": 1000,
        "random_seed": SEED,
        "verbose": False,
        "cat_features": cat_cols,
        "early_stopping_rounds": 50,
        "task_type": "GPU",
    }
    model = cb.CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    preds = model.predict_proba(X_va)[:, 1]
    return log_loss(y_va, preds)

logger.info("実験A: 手動CatBoostのOptuna探索開始...")
study_manual_cat = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_manual_cat.optimize(objective_manual_cat, n_trials=30)

logger.info(f"実験A Best Log Loss: {study_manual_cat.best_value:.6f}")
logger.info(f"実験A Best Params: {study_manual_cat.best_params}")

[2026-08-08 16:14:17] [INFO] 実験A 特徴量数: 314, カテゴリ列: ['入社区分', '専攻分野', '採用経路', '性別', '初期職種', '初期勤務地', '初期役割']


INFO:14_catboost_ablation_autogluon_hpo_bk:実験A 特徴量数: 314, カテゴリ列: ['入社区分', '専攻分野', '採用経路', '性別', '初期職種', '初期勤務地', '初期役割']


[2026-08-08 16:14:17] [INFO] 実験A: 手動CatBoostのOptuna探索開始...


INFO:14_catboost_ablation_autogluon_hpo_bk:実験A: 手動CatBoostのOptuna探索開始...


[2026-08-08 16:27:32] [INFO] 実験A Best Log Loss: 0.550679


INFO:14_catboost_ablation_autogluon_hpo_bk:実験A Best Log Loss: 0.550679


[2026-08-08 16:27:32] [INFO] 実験A Best Params: {'depth': 6, 'learning_rate': 0.07217714211643014, 'l2_leaf_reg': 4.2117355396513725, 'border_count': 104, 'bagging_temperature': 0.7323080468306031, 'random_strength': 5.288387671549879}


INFO:14_catboost_ablation_autogluon_hpo_bk:実験A Best Params: {'depth': 6, 'learning_rate': 0.07217714211643014, 'l2_leaf_reg': 4.2117355396513725, 'border_count': 104, 'bagging_temperature': 0.7323080468306031, 'random_strength': 5.288387671549879}


In [55]:
# ベストパラメータで深い再学習
best_params_manual = study_manual_cat.best_params
final_manual_cat = cb.CatBoostClassifier(
    **best_params_manual,
    iterations=3000,
    random_seed=SEED,
    verbose=False,
    cat_features=cat_cols,
    early_stopping_rounds=100,
    task_type="GPU",
)
final_manual_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)

manual_cat_val_preds = final_manual_cat.predict_proba(X_va)[:, 1]
manual_cat_val_score = log_loss(y_va, manual_cat_val_preds)
logger.info(f"実験A 最終検証Log Loss: {manual_cat_val_score:.6f}")
logger.info(f"(参考) 13_のAutoGluon CatBoost検証Log Loss: 0.567941")

X_test_manual = test_features[feature_cols].fillna(-999)
manual_cat_test_preds = final_manual_cat.predict_proba(X_test_manual)[:, 1]

sub_manual_cat = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: manual_cat_test_preds})
sub_manual_cat.to_csv(SUBMISSION_MANUAL_CAT_PATH, index=False, header=False)
logger.info(f"実験A 提出ファイル保存: {SUBMISSION_MANUAL_CAT_PATH}")

print(f"\n■ 実験A（手動CatBoost, 単一時系列ホールドアウト）検証Log Loss: {manual_cat_val_score:.6f}")
print(f"■ (参考) 13_のAutoGluon CatBoost検証Log Loss: 0.567941")
print(f"■ 提出ファイル: {SUBMISSION_MANUAL_CAT_PATH}")

[2026-08-08 16:27:42] [INFO] 実験A 最終検証Log Loss: 0.550679


INFO:14_catboost_ablation_autogluon_hpo_bk:実験A 最終検証Log Loss: 0.550679


[2026-08-08 16:27:42] [INFO] (参考) 13_のAutoGluon CatBoost検証Log Loss: 0.567941


INFO:14_catboost_ablation_autogluon_hpo_bk:(参考) 13_のAutoGluon CatBoost検証Log Loss: 0.567941


[2026-08-08 16:27:42] [INFO] 実験A 提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_manual_catboost_holdout.csv


INFO:14_catboost_ablation_autogluon_hpo_bk:実験A 提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_manual_catboost_holdout.csv



■ 実験A（手動CatBoost, 単一時系列ホールドアウト）検証Log Loss: 0.550679
■ (参考) 13_のAutoGluon CatBoost検証Log Loss: 0.567941
■ 提出ファイル: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_manual_catboost_holdout.csv


## 4. 実験B: AutoGluonでCatBoostの複数バリエーション + 長めのtime_limitで再学習

**疑問2への回答**: `hyperparameter_tune_kwargs`によるAutoGluon内蔵HPOはバージョンによってAPIが変わりやすく、
ローカルで動作確認ができない（本環境にAutoGluon未インストールのため）。そこで、より安定した方法として、
`hyperparameters`に**CatBoostの複数の固定パラメータ候補をリストで渡す**方式を採用する
（AutoGluonが各候補を学習し、Weighted Ensembleの構築時にどれを採用するか自動で判断する）。
内部バギング（`num_bag_folds=0, num_stack_levels=0`）は13_から変更せず、過学習リスクの再燃を避ける。

**疑問4（運用改善）への対応**: 最初からWeighted EnsembleとSingle Best Modelの両方を保存する。

In [56]:
# CatBoostの複数バリエーション（13_のデフォルト設定 + 実験Aで探索した近傍 + 深め/浅めのバリエーション）
catboost_variants = [
    {},  # AutoGluonの既定（13_で最良だった設定）
    {"depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 3.0},
    {"depth": 6, "learning_rate": 0.02, "l2_leaf_reg": 5.0},
    {"depth": 8, "learning_rate": 0.015, "l2_leaf_reg": 1.0},
]

hyperparameters = {
    "CAT": catboost_variants,
    "GBM": {},
    "XGB": {},
}

predictor_hpo = TabularPredictor(
    label=TARGET_COL,
    problem_type="binary",
    eval_metric="log_loss",
    path=str(AUTOGLUON_MODEL_DIR),
)

predictor_hpo.fit(
    train_data=ag_train_data,
    tuning_data=ag_tuning_data,
    hyperparameters=hyperparameters,
    num_bag_folds=0,
    num_stack_levels=0,
    num_gpus=1,
    time_limit=5400,  # 13_の1時間から1.5時間に延長
    verbosity=2,
)

logger.info("実験B AutoGluon学習完了")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          8
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.55/14.56 GB
Total GPU Memory:   Free: 14.55 GB, Allocated: 0.02 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       46.52 GB / 50.99 GB (91.2%)
Disk Space Avail:   178.39 GB / 235.68 GB (75.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on 

[2026-08-08 16:29:33] [INFO] 実験B AutoGluon学習完了


INFO:14_catboost_ablation_autogluon_hpo_bk:実験B AutoGluon学習完了


In [57]:
leaderboard_hpo = predictor_hpo.leaderboard(ag_tuning_data, silent=True)
logger.info("=" * 60)
logger.info("実験B Leaderboard")
logger.info("=" * 60)
logger.info("\n" + leaderboard_hpo.to_string())

score_col_hpo = "score_test" if "score_test" in leaderboard_hpo.columns else "score_val"
best_model_name_hpo = predictor_hpo.model_best
best_score_hpo = leaderboard_hpo.loc[leaderboard_hpo["model"] == best_model_name_hpo, score_col_hpo].values[0]
logger.info(f"実験B Best model: {best_model_name_hpo}, Log Loss: {-best_score_hpo:.6f}")
logger.info(f"(参考) 13_のWeighted Ensemble検証Log Loss: 0.567022 / CatBoost単体: 0.567941")

leaderboard_hpo

[2026-08-08 16:29:33] [INFO] ============================================================


INFO:14_catboost_ablation_autogluon_hpo_bk:============================================================


[2026-08-08 16:29:33] [INFO] 実験B Leaderboard


INFO:14_catboost_ablation_autogluon_hpo_bk:実験B Leaderboard


[2026-08-08 16:29:33] [INFO] ============================================================


...

[2026-08-08 16:29:33] [INFO] 実験B Best model: WeightedEnsemble_L2, Log Loss: 0.552597


INFO:14_catboost_ablation_autogluon_hpo_bk:実験B Best model: WeightedEnsemble_L2, Log Loss: 0.552597


[2026-08-08 16:29:33] [INFO] (参考) 13_のWeighted Ensemble検証Log Loss: 0.567022 / CatBoost単体: 0.567941


INFO:14_catboost_ablation_autogluon_hpo_bk:(参考) 13_のWeighted Ensemble検証Log Loss: 0.567022 / CatBoost単体: 0.567941


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_4,-0.552597,-0.552597,log_loss,0.025864,0.029047,56.855872,0.025864,0.029047,56.855872,1,True,5
1,WeightedEnsemble_L2,-0.552597,-0.552597,log_loss,0.029039,0.029650,56.868819,0.003175,0.000603,0.012947,2,True,7
2,CatBoost_3,-0.558117,-0.558117,log_loss,0.025839,0.024374,22.505672,0.025839,0.024374,22.505672,1,True,4
3,CatBoost,-0.567941,-0.567941,log_loss,0.026719,0.019976,10.817392,0.026719,0.019976,10.817392,1,True,2
4,CatBoost_2,-0.569613,-0.569613,log_loss,0.024577,0.030217,9.142103,0.024577,0.030217,9.142103,1,True,3
5,LightGBM,-0.584156,-0.584156,log_loss,0.008654,0.006860,4.013282,0.008654,0.006860,4.013282,1,True,1
6,XGBoost,-0.590151,-0.590151,log_loss,0.024275,0.013728,5.689976,0.024275,0.013728,5.689976,1,True,6


In [58]:
# Weighted Ensemble での提出ファイル作成
X_test = test_features.copy()
proba_df_hpo = predictor_hpo.predict_proba(X_test)
positive_col_hpo = 1 if 1 in proba_df_hpo.columns else proba_df_hpo.columns[-1]
test_preds_hpo_ensemble = proba_df_hpo[positive_col_hpo].values

sub_hpo_ensemble = pd.DataFrame({ID_COL: X_test.index, TARGET_COL: test_preds_hpo_ensemble})
sub_hpo_ensemble.to_csv(SUBMISSION_AG_ENSEMBLE_PATH, index=False, header=False)
logger.info(f"実験B Weighted Ensemble提出ファイル保存: {SUBMISSION_AG_ENSEMBLE_PATH}")

# 単体最良モデルでの提出ファイルも最初から作成（疑問4への対応）
single_model_leaderboard_hpo = leaderboard_hpo[~leaderboard_hpo["model"].str.contains("WeightedEnsemble", na=False)]
single_model_leaderboard_hpo = single_model_leaderboard_hpo.sort_values(score_col_hpo, ascending=False)
single_best_model_hpo = single_model_leaderboard_hpo["model"].iloc[0]
single_best_score_hpo = single_model_leaderboard_hpo[score_col_hpo].iloc[0]

proba_single_hpo = predictor_hpo.predict_proba(X_test, model=single_best_model_hpo)
test_preds_single_hpo = proba_single_hpo[positive_col_hpo].values

safe_model_name_hpo = "".join(c if c.isalnum() or c == "_" else "_" for c in single_best_model_hpo)
SUBMISSION_AG_SINGLE_BEST_PATH = Path(SUBMISSION_AG_SINGLE_BEST_PATH_TEMPLATE.format(model=safe_model_name_hpo))

sub_hpo_single = pd.DataFrame({ID_COL: X_test.index, TARGET_COL: test_preds_single_hpo})
sub_hpo_single.to_csv(SUBMISSION_AG_SINGLE_BEST_PATH, index=False, header=False)
logger.info(f"実験B 単体最良モデル({single_best_model_hpo})提出ファイル保存: {SUBMISSION_AG_SINGLE_BEST_PATH}")

print(f"\n■ 実験B Weighted Ensemble  : {best_model_name_hpo} (Log Loss {-best_score_hpo:.6f}) -> {SUBMISSION_AG_ENSEMBLE_PATH}")
print(f"■ 実験B 単体最良モデル      : {single_best_model_hpo} (Log Loss {-single_best_score_hpo:.6f}) -> {SUBMISSION_AG_SINGLE_BEST_PATH}")

[2026-08-08 16:29:34] [INFO] 実験B Weighted Ensemble提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_autogluon_ensemble.csv


INFO:14_catboost_ablation_autogluon_hpo_bk:実験B Weighted Ensemble提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_autogluon_ensemble.csv


[2026-08-08 16:29:34] [INFO] 実験B 単体最良モデル(CatBoost_4)提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_autogluon_single_best_CatBoost_4.csv


INFO:14_catboost_ablation_autogluon_hpo_bk:実験B 単体最良モデル(CatBoost_4)提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_autogluon_single_best_CatBoost_4.csv



■ 実験B Weighted Ensemble  : WeightedEnsemble_L2 (Log Loss 0.552597) -> /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_autogluon_ensemble.csv
■ 実験B 単体最良モデル      : CatBoost_4 (Log Loss 0.552597) -> /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_14_catboost_ablation_autogluon_hpo_bk_autogluon_single_best_CatBoost_4.csv


## 5. まとめ

| 実験 | 手法 | 検証Log Loss（単一時系列ホールドアウト） | 提出ファイル |
|---|---|---|---|
| 13_（参考） | AutoGluon Weighted Ensemble | 0.567022 | `20260808_13_autogluon_prototype_autogluon.csv`（Public 0.575345） |
| 13_（参考） | AutoGluon CatBoost単体 | 0.567941 | `20260808_13_autogluon_prototype_single_best_CatBoost.csv`（Public 0.574165、★これまでの最良） |
| **実験A** | 手動CatBoost（Optuna, 30試行） | 実行結果を記入 | `SUBMISSION_MANUAL_CAT_PATH` |
| **実験B** | AutoGluon（CatBoost複数候補 + time_limit延長） Weighted Ensemble | 実行結果を記入 | `SUBMISSION_AG_ENSEMBLE_PATH` |
| **実験B** | 同上 単体最良モデル | 実行結果を記入 | `SUBMISSION_AG_SINGLE_BEST_PATH` |

### 疑問への回答（実行後に埋める）

1. **なぜAutoGluonの単体CatBoostが12_の手動CatBoードを上回ったのか**
   → 実験Aの検証Log Lossが0.567941（13_のAutoGluon CatBoost）に近ければ、
   「検証方式（単一時系列ホールドアウト）と特徴量の単純化」が主因。大きく劣っていれば、
   AutoGluon固有のゼロショット既定パラメータや内部前処理（特にカテゴリ変数の扱い）が効いていることになる。

2. **`time_limit`を伸ばす・複数パラメータ候補を与えることでさらに改善するか**
   → 実験Bの結果が13_（0.567022 / 0.567941）を上回るか確認する。

3. **単一時系列ホールドアウトは他のモデルでも機能するか**
   → 実験Aも同じ検証方式を使っているため、Public提出後にそのギャップ（実験A提出のPublicスコア - 検証スコア）を
   これまでの結果（13_: 0.0062〜0.0083）と比較する。

4. **アンサンブルは単体最良モデルより良いか**
   → 実験Bで両方の提出ファイルを用意したので、Publicスコアを比較する。

3つとも提出してPublicスコアが揃ったら、`submit_result_report.md`に追記する。